# Log Anomaly Detection

Is notebook ko step by step chalayein. Hum sample log file padhenge, unusual messages dhoondhenge, charts dekhenge, aur model save karenge.

**Frontend ke baare mein:** website abhi har upload ko usi file ke andar analyze karti hai. Notebook mein save hua model frontend khud se load nahi karta.

HDFS sample mein sahi anomaly labels nahi hain. Isliye is sample se accuracy, precision, ya recall nahi mil sakte. “Flagged” ka matlab bas itna hai ki message file ke baaki messages se alag laga—yeh pakka incident nahi hai.

## Shuru karne se pehle

Colab ya Jupyter mein **Run All** dabayein. Agar apni file use karni hai, step 2 ke baad `DATA_PATH` mein uska path dein. CSV mein `EventTemplate` ya `message` jaisa column rakhein; normal/anomaly ka sachcha label ho to `label` column add karein (`0 = normal, 1 = anomaly`).

In [35]:
# Colab/Jupyter mein chahiye hone wale packages install karein.
%pip -q install pandas scikit-learn plotly joblib nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1 — Libraries load karein

Yeh libraries file padhne, text ko numbers mein badalne, models chalane aur charts banane ke kaam aati hain.

In [18]:
from pathlib import Path
import time
import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

## Step 2 — File aur settings chunein

`CONTAMINATION` yahan sirf expected flagged rate hai; yeh accuracy nahi batata. Apni file ka path `DATA_PATH` mein rakh sakte hain.

In [20]:
CONTAMINATION = 0.10
RANDOM_STATE = 42
sample_paths = (
    Path("loghub/HDFS/HDFS_2k.log_structured.csv"),
    Path("../loghub/HDFS/HDFS_2k.log_structured.csv"),
)
DATA_PATH = next((path for path in sample_paths if path.is_file()), sample_paths[0])

# Agar sample file yahan nahi mili, Colab mein apni CSV ya log file upload karein.
if not DATA_PATH.is_file():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "DATA_PATH mein apni CSV, .log, ya .txt file ka path dein."
        ) from exc

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Ek baar mein sirf ek file upload karein.")
    DATA_PATH = Path(next(iter(uploaded)))

print("File:", DATA_PATH.resolve())

File: C:\Users\hp\OneDrive\Desktop\Projects\log_anomaly\loghub\HDFS\HDFS_2k.log_structured.csv


## Step 3 — File padhein

CSV ko table ke roop mein padhenge. .log ya .txt mein har line ko ek event maanenge. Pehli kuch lines dekhkar check karein ki file sahi khuli hai.

In [21]:
if DATA_PATH.suffix.lower() == ".csv":
    data = pd.read_csv(DATA_PATH)
else:
    lines = DATA_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    data = pd.DataFrame({"message": lines})

print(f"Rows: {len(data):,}")
print("Columns:", ", ".join(map(str, data.columns)))
display(data.head())

Rows: 2,000
Columns: LineId, Date, Time, Pid, Level, Component, Content, EventId, EventTemplate


,LineId,Date,Time,Pid,Level,Component,Content,EventId,EventTemplate
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...,E10,PacketResponder <*> for block blk_<*> terminating
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...,E10,PacketResponder <*> for block blk_<*> terminating
2,3,81109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...,E10,PacketResponder <*> for block blk_<*> terminating
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...,E10,PacketResponder <*> for block blk_<*> terminating


## Step 4 — Har log line ka text chunen

Structured HDFS file mein `EventTemplate` raw `Content` se behtar hai: ismein badalte block IDs pehle se placeholders bane hote hain. Blank lines hata kar, agar labels hon, unhe alag rakhenge.

In [22]:
# Template ko pehle dekhein; warna message ya content use karein.
priority = ("eventtemplate", "event_template", "message", "msg", "content", "log", "text")
columns = {str(column).strip().lower(): column for column in data.columns}
message_column = next((columns[name] for name in priority if name in columns), None)

if message_column is None:
    raise ValueError("CSV mein EventTemplate, message, Content ya log column chahiye.")

messages = data[message_column].fillna("").astype(str).str.strip()
keep = messages.ne("")
data = data.loc[keep].reset_index(drop=True)
messages = messages.loc[keep].reset_index(drop=True)

print("Text column:", message_column)
print(f"Events: {len(messages):,}")
print(f"Alag-alag messages: {messages.nunique():,}")

Text column: EventTemplate
Events: 2,000
Alag-alag messages: 14


In [23]:
# Labels sirf performance naapne ke liye hain; detector ko train karne ke liye zaroori nahi.
label_names = {"label", "is_anomaly", "anomaly", "ground_truth", "target"}
label_column = next(
    (column for column in data.columns if str(column).strip().lower() in label_names),
    None,
)
labels = None

if label_column is None:
    print("Label column nahi mila. Accuracy/precision/recall report nahi ki ja sakti.")
else:
    values = data[label_column].astype(str).str.strip().str.lower()
    positive = {"1", "1.0", "true", "yes", "anomaly", "abnormal"}
    negative = {"0", "0.0", "false", "no", "normal"}

    if values.isin(positive | negative).all():
        labels = values.isin(positive).to_numpy()
        print(f"Labels mile: {int(labels.sum())} anomaly, {int((~labels).sum())} normal")
    else:
        print("Label values pehchane nahi gaye; 1/0 ya anomaly/normal use karein.")

Label column nahi mila. Accuracy/precision/recall report nahi ki ja sakti.


## Step 5 — Text ko numbers mein badlein

Model seedhe sentences nahi samajhta. TF-IDF har message ke words ko numbers mein badalta hai. Same vectorizer ko aage model aur naye messages dono ke liye use karna zaroori hai.

In [24]:
vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
features = vectorizer.fit_transform(messages)

print(f"Feature matrix: {features.shape[0]:,} events × {features.shape[1]:,} words")

Feature matrix: 2,000 events × 39 words


## Step 6 — Pehla model: Isolation Forest

Yeh project ka main detector hai. Yeh aise text patterns dhoondhta hai jo baaki messages se alag hain. Score jitna bada, message utna unusual.

In [25]:
detectors = {}
scores = {}
flags = {}
fit_seconds = {}

def fit_and_score(name, model, fit_features, events_to_score):
    start = time.perf_counter()
    model.fit(fit_features)
    score = -model.decision_function(events_to_score)
    fit_seconds[name] = time.perf_counter() - start
    scores[name] = np.asarray(score)

    # Top expected fraction ko flag karein; barabar score hon to ginti badh sakti hai.
    cutoff = np.quantile(scores[name], 1 - CONTAMINATION)
    flags[name] = scores[name] >= cutoff
    detectors[name] = model

    return int(flags[name].sum())

isolation_forest = IsolationForest(
    n_estimators=100,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

count = fit_and_score("Isolation Forest", isolation_forest, features, features)
print(f"Isolation Forest ne {count:,} events flag kiye.")

Isolation Forest ne 223 events flag kiye.


## Step 7 — Doosra model: Local Outlier Factor

LOF aas-paas ke messages se tulna karta hai. HDFS mein ek hi template kai baar aata hai; isliye ise unique templates par fit karke phir har event ka score nikalte hain.

In [26]:
unique_rows = ~messages.duplicated().to_numpy()
unique_features = features[unique_rows]

lof = LocalOutlierFactor(
    n_neighbors=min(20, unique_features.shape[0] - 1),
    contamination=CONTAMINATION,
    metric="cosine",
    novelty=True,
    n_jobs=1,
)

count = fit_and_score("Local Outlier Factor", lof, unique_features, features)
print(f"Local Outlier Factor ne {count:,} events flag kiye.")

Local Outlier Factor ne 407 events flag kiye.


## Step 8 — Teesra model: One-Class SVM

Yeh bhi unlabeled data mein aam pattern seekhta hai. Alag algorithm ke results dekhne se pata chalta hai ki kaunse events har model ko unusual lagte hain.

In [27]:
one_class_svm = OneClassSVM(kernel="linear", nu=CONTAMINATION)
count = fit_and_score("One-Class SVM", one_class_svm, features, features)
print(f"One-Class SVM ne {count:,} events flag kiye.")

One-Class SVM ne 319 events flag kiye.


## Step 9 — Seedha baseline aur comparison

Baseline sirf yeh dekhta hai ki kaunsa template kitni baar aaya. Rare template ko unusual maanta hai. Yeh simple comparison hai, accuracy ka saboot nahi.

In [28]:
template_counts = messages.value_counts()
frequency_score = messages.map(lambda text: 1 / template_counts[text]).to_numpy()
scores["Rare-template baseline"] = frequency_score
flags["Rare-template baseline"] = frequency_score >= np.quantile(
    frequency_score, 1 - CONTAMINATION
)
fit_seconds["Rare-template baseline"] = 0.0

comparison = pd.DataFrame([
    {
        "model": name,
        "time_seconds": round(fit_seconds[name], 3),
        "flagged_events": int(flags[name].sum()),
        "flagged_percent": round(100 * flags[name].mean(), 2),
    }
    for name in scores
]).sort_values("flagged_percent", ascending=False)

display(comparison)
print("Flag count zyada hona model ke behtar hone ka matlab nahi.")

,model,time_seconds,flagged_events,flagged_percent
1,Local Outlier Factor,0.064,407,20.35
2,One-Class SVM,0.133,319,15.95
3,Rare-template baseline,0.000,304,15.20
0,Isolation Forest,1.493,223,11.15


Flag count zyada hona model ke behtar hone ka matlab nahi.


## Step 10 — Chart: aam log patterns

Yahan sabse zyada aane wale templates dikhte hain. Hover karke count dekhein; rare template akela anomaly prove nahi karta.

In [30]:
chart_data = template_counts.head(12).sort_values().rename_axis("template").reset_index(name="events")
px.bar(
    chart_data,
    x="events",
    y="template",
    orientation="h",
    title="Sabse aam 12 log templates",
    labels={"events": "Events", "template": "Log template"},
)

## Step 11 — Chart: har model ne kitne events flag kiye

Models ke flag rate alag ho sakte hain. Yeh chart sirf unka behavior dikhata hai—kaun sahi hai, woh nahi.

In [31]:
display(px.bar(
    comparison.sort_values("flagged_percent"),
    x="flagged_percent",
    y="model",
    orientation="h",
    title="Har model ne kitne percent events flag kiye?",
    labels={"flagged_percent": "Flagged events (%)", "model": "Model"},
))

## Step 12 — Chart: Isolation Forest ke scores

High score ka matlab hai ki model ko message zyada unusual laga. Use log context ke saath check karein.

In [32]:
score_data = pd.DataFrame({
    "score": scores["Isolation Forest"],
    "result": np.where(flags["Isolation Forest"], "Flagged", "Not flagged"),
})
display(px.histogram(
    score_data,
    x="score",
    color="result",
    barmode="overlay",
    opacity=0.75,
    title="Isolation Forest anomaly scores",
    labels={"score": "Anomaly score", "result": "Result"},
))

## Step 13 — Labels hon tabhi metrics dekhein

`label` column mein 0 normal aur 1 anomaly dene par yeh step chalega. Detector sirf training set ke normal events par fit hota hai; test set ke labels model ko train karne mein nahi diye jaate. Yeh same dataset ka holdout hai—dusre log type par sahi chalne ka proof nahi. Uske liye alag source/dataset par bhi test karein.

In [33]:
if labels is None:
    print("Skip: file mein sahi 0/1 ground-truth labels nahi mile.")
elif len(np.unique(labels)) < 2:
    print("Skip: metrics ke liye normal aur anomaly dono labels chahiye.")
else:
    train_idx, test_idx = train_test_split(
        np.arange(len(labels)),
        test_size=0.25,
        random_state=RANDOM_STATE,
        stratify=labels,
    )
    normal_idx = train_idx[~labels[train_idx]]
    normal_text = messages.iloc[normal_idx].reset_index(drop=True)

    if len(normal_text) < 3 or normal_text.nunique() < 3:
        print("Skip: training data mein kam-se-kam 3 alag normal message patterns chahiye.")
    else:
        holdout_vectorizer = TfidfVectorizer(token_pattern=r"(?u)\b\w+\b")
        normal_features = holdout_vectorizer.fit_transform(normal_text)
        test_features = holdout_vectorizer.transform(messages.iloc[test_idx])
        unique_normal = normal_features[~normal_text.duplicated().to_numpy()]

        test_models = {
            "Isolation Forest": (
                IsolationForest(n_estimators=100, contamination="auto", random_state=RANDOM_STATE, n_jobs=1),
                normal_features,
            ),
            "Local Outlier Factor": (
                LocalOutlierFactor(
                    n_neighbors=min(20, unique_normal.shape[0] - 1),
                    contamination="auto", metric="cosine", novelty=True, n_jobs=1,
                ),
                unique_normal,
            ),
            "One-Class SVM": (OneClassSVM(kernel="linear", nu=0.01), normal_features),
        }

        y_true = labels[test_idx]
        rows = []
        for name, (model, fit_data) in test_models.items():
            model.fit(fit_data)
            normal_scores = -model.decision_function(normal_features)
            test_scores = -model.decision_function(test_features)
            cutoff = np.quantile(normal_scores, 0.99)
            y_pred = test_scores >= cutoff

            precision, recall, f1, _ = precision_recall_fscore_support(
                y_true, y_pred, average="binary", zero_division=0
            )
            rows.append({
                "model": name,
                "accuracy": accuracy_score(y_true, y_pred),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
                "precision": precision,
                "recall": recall,
                "f1": f1,
                "average_precision": average_precision_score(y_true, test_scores),
                "test_events": len(y_true),
            })

        metrics = pd.DataFrame(rows).sort_values("f1", ascending=False)
        display(metrics.round(3))
        print("Threshold normal training logs ke top 1% scores par set kiya gaya hai.")
        print("Accuracy ko precision, recall aur balanced accuracy ke saath padhein.")

Skip: file mein sahi 0/1 ground-truth labels nahi mile.


## Step 14 — Model ko save karein

Yeh fitted Isolation Forest aur wahi TF-IDF vectorizer ek file mein save honge. Isse dobara load karke use kiya ja sakta hai, lekin current frontend ise automatically load nahi karta.

In [34]:
model_dir = next(
    (path for path in (Path("backend/models"), Path("../backend/models"), Path("models"), Path("/content")) if path.is_dir()),
    Path.cwd(),
)
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "log_anomaly_isolation_forest.joblib"

joblib.dump({
    "model": detectors["Isolation Forest"],
    "vectorizer": vectorizer,
    "message_column": message_column,
    "trained_events": len(messages),
    "contamination": CONTAMINATION,
}, model_path)

print("Model save hua:", model_path.resolve())

Model save hua: C:\Users\hp\OneDrive\Desktop\Projects\log_anomaly\backend\models\log_anomaly_isolation_forest.joblib


## Result ko kaise samjhein?

- HDFS sample mein 2,000 events aur 14 templates hain, par true anomaly labels nahi hain. Isliye is file se model ki accuracy prove nahi hoti.
- Flagged events ko manually dekhein. Rare ya alag message kabhi normal bhi ho sakta hai.
- Apne system par use karne se pehle us system ke logs par check karein. Sahi labels mil sakein to alag test data par precision, recall aur F1 compare karein.
- Yahan ka saved model notebook ke liye hai. Website abhi har uploaded file ko alag se analyze karti hai; website ko is saved model se jodne ke liye backend mein alag integration chahiye.